In [12]:
# =========================
# STEP 0 — IMPORTS, PATH, FOLDERS
# =========================
import os, re, numpy as np, pandas as pd

try:
    from IPython.display import display
except ImportError:
    def display(x): print(x.head() if hasattr(x, "head") else x)

# Your paths (UNCHANGED)
DATA_PATH   = r"C:\Users\dusik\OneDrive\Desktop\Y2S1\Data\raw\diabetic_data.csv"
OUTPUTS_DIR = r"C:\Users\dusik\OneDrive\Desktop\Y2S1\Results\Outputs"
os.makedirs(OUTPUTS_DIR, exist_ok=True)

# Prefer the encoded dataset produced earlier (if present)
ENCODED_PATH = os.path.join(OUTPUTS_DIR, "diabetic_encoded.csv")

# Common config (keep IDs/targets untouched)
TARGET_COL   = "readmitted"
DERIVED_TGT  = "readmitted_30d"
ID_COLS      = ["encounter_id", "patient_nbr"]
PLACEHOLDERS = ["?", "Unknown/Invalid", "Unknown", "None", "N/A", "NA", "NULL", "Not Available"]

def shape_note(tag, df): print(f"{tag:<28} -> shape: {df.shape}")
print("Outputs will be saved to:", OUTPUTS_DIR)

Outputs will be saved to: C:\Users\dusik\OneDrive\Desktop\Y2S1\Results\Outputs


In [13]:
# =========================
# STEP 1 — LOAD DATA (prefer ENCODED), BASIC STANDARDIZE
# =========================
if os.path.exists(ENCODED_PATH):
    df = pd.read_csv(ENCODED_PATH)
    shape_note("Loaded ENCODED dataset", df)
else:
    if not os.path.exists(DATA_PATH):
        raise FileNotFoundError(f"Could not find dataset at:\n{DATA_PATH}")
    df = pd.read_csv(DATA_PATH)
    shape_note("Loaded RAW dataset", df)
    # Standardize placeholder missings to NaN; trim whitespace in objects
    df = df.replace(PLACEHOLDERS, np.nan)
    for c in df.select_dtypes(include=["object"]).columns:
        df[c] = df[c].astype(str).str.strip()

display(df.head())

Loaded ENCODED dataset       -> shape: (101766, 111)


,encounter_id,patient_nbr,readmitted,gender,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,medical_specialty,num_lab_procedures,num_procedures,num_medications,number_outpatient,number_emergency,number_inpatient,diag_1,diag_2,diag_3,number_diagnoses,max_glu_serum,A1Cresult,change,diabetesMed,race_Asian,race_Caucasian,race_Hispanic,race_Other,race_nan,age_[10-20),age_[20-30),age_[30-40),age_[40-50),age_[50-60),age_[60-70),age_[70-80),age_[80-90),age_[90-100),weight_[0-25),weight_[100-125),weight_[125-150),weight_[150-175),weight_[175-200),weight_[25-50),weight_[50-75),weight_[75-100),weight_nan,payer_code_CH,payer_code_CM,payer_code_CP,payer_code_DM,payer_code_FR,payer_code_HM,payer_code_MC,payer_code_MD,payer_code_MP,payer_code_OG,payer_code_OT,payer_code_PO,payer_code_SI,payer_code_SP,payer_code_UN,payer_code_WC,payer_code_nan,metformin_No,metformin_Steady,metformin_Up,repaglinide_No,repaglinide_Steady,repaglinide_Up,nateglinide_No,nateglinide_Steady,nateglinide_Up,chlorpropamide_No,chlorpropamide_Steady,chlorpropamide_Up,glimepiride_No,glimepiride_Steady,glimepiride_Up,acetohexamide_Steady,glipizide_No,glipizide_Steady,glipizide_Up,glyburide_No,glyburide_Steady,glyburide_Up,tolbutamide_Steady,pioglitazone_No,pioglitazone_Steady,pioglitazone_Up,rosiglitazone_No,rosiglitazone_Steady,rosiglitazone_Up,acarbose_No,acarbose_Steady,acarbose_Up,miglitol_No,miglitol_Steady,miglitol_Up,troglitazone_Steady,tolazamide_Steady,tolazamide_Up,insulin_No,insulin_Steady,insulin_Up,glyburide-metformin_No,glyburide-metformin_Steady,glyburide-metformin_Up,glipizide-metformin_Steady,glimepiride-pioglitazone_Steady,metformin-rosiglitazone_Steady,metformin-pioglitazone_Steady
0,2278392,8222157,NO,0.0,6,25,1,1,0.001562,41,0,1,0,0,0,0.000934,0.003518,0.013983,1,NaN,NaN,0,0,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,True,False,False,True,False,False,True,False,False,True,False,False,True,False,False,False,True,False,False,True,False,False,False,True,False,False,True,False,False,True,False,False,True,False,False,False,False,False,True,False,False,True,False,False,False,False,False,False
1,149190,55629189,>30,0.0,1,1,7,3,0.490822,59,0,18,0,0,0,0.018562,0.014966,0.000678,9,NaN,NaN,1,1,False,True,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,True,False,False,True,False,False,True,False,False,True,False,False,True,False,False,False,True,False,False,True,False,False,False,True,False,False,True,False,False,True,False,False,True,False,False,False,False,False,False,False,True,True,False,False,False,False,False,False
2,64410,86047875,NO,0.0,1,1,7,2,0.490822,11,5,13,2,0,1,0.002801,0.059656,0.000364,6,NaN,NaN,0,1,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,True,False,False,True,False,False,True,False,False,True,False,False,True,False,False,False,False,True,False,True,False,False,False,True,False,False,True,False,False,True,False,False,True,False,False,False,False,False,True,False,False,True,False,False,False,False,False,False
3,500364,82442376,NO,1.0,1,1,7,2,0.490822,44,1,16,0,0,0,0.005061,0.000364,0.023161,7,NaN,NaN,1,1,False,True,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,True,False,False,True,False,False,True,False,False,True,False,False,True,False,False,False,True,False,False,True,False,False,F

In [14]:
# =========================
# STEP 2 — FEATURE ENGINEERING (safe, no leakage from target)
# =========================
fe_df = df.copy()
added_cols = []

# Helper: safe division avoiding zero/NaN
def sdiv(a, b):
    return np.where((b==0) | pd.isna(b), np.nan, a / b)

# 2A) Age midpoint from bracket like "[70-80)"
if "age" in fe_df.columns and fe_df["age"].dtype == "object":
    def age_mid(x):
        nums = re.findall(r"\d+", str(x))
        return (int(nums[0]) + int(nums[1]))/2 if len(nums)>=2 else np.nan
    fe_df["fe_age_mid"] = fe_df["age"].apply(age_mid)
    added_cols.append("fe_age_mid")
    # Senior flag
    fe_df["fe_is_senior"] = (fe_df["fe_age_mid"] >= 65).astype("Int64")
    added_cols.append("fe_is_senior")

# 2B) Visit aggregation
for need in ["number_outpatient", "number_emergency", "number_inpatient"]:
    if need not in fe_df.columns:
        fe_df[need] = fe_df.get(need, 0)
if set(["number_outpatient","number_emergency","number_inpatient"]).issubset(fe_df.columns):
    fe_df["fe_total_visits"] = (
        fe_df["number_outpatient"].fillna(0)
        + fe_df["number_emergency"].fillna(0)
        + fe_df["number_inpatient"].fillna(0)
    )
    added_cols.append("fe_total_visits")

# 2C) Intensity ratios (per-day rates)
if "time_in_hospital" in fe_df.columns:
    if "num_lab_procedures" in fe_df.columns:
        fe_df["fe_labs_per_day"] = sdiv(fe_df["num_lab_procedures"], fe_df["time_in_hospital"])
        added_cols.append("fe_labs_per_day")
    if "num_medications" in fe_df.columns:
        fe_df["fe_meds_per_day"] = sdiv(fe_df["num_medications"], fe_df["time_in_hospital"])
        added_cols.append("fe_meds_per_day")

# 2D) Procedure–Medication balance
if "num_medications" in fe_df.columns and "num_procedures" in fe_df.columns:
    fe_df["fe_procs_to_meds"] = sdiv(fe_df["num_procedures"], fe_df["num_medications"])
    added_cols.append("fe_procs_to_meds")

# 2E) Ordinal encodings (if still strings here; if already encoded earlier this will be skipped)
if "insulin" in fe_df.columns and fe_df["insulin"].dtype == "object":
    fe_df["fe_insulin_trend"] = fe_df["insulin"].map({"No":0, "Down":1, "Steady":2, "Up":3}).astype("Int64")
    added_cols.append("fe_insulin_trend")

if "change" in fe_df.columns and fe_df["change"].dtype == "object":
    fe_df["fe_change"] = fe_df["change"].map({"No":0, "Ch":1}).astype("Int64")
    added_cols.append("fe_change")

if "diabetesMed" in fe_df.columns and fe_df["diabetesMed"].dtype == "object":
    fe_df["fe_diabetesMed"] = fe_df["diabetesMed"].map({"No":0, "Yes":1}).astype("Int64")
    added_cols.append("fe_diabetesMed")

# 2F) Clinical result tiers to ordinal (if not already numeric)
if "A1Cresult" in fe_df.columns and fe_df["A1Cresult"].dtype == "object":
    fe_df["fe_A1C_ordinal"] = fe_df["A1Cresult"].map({"None":0, "Norm":1, ">7":2, ">8":3}).astype("Int64")
    added_cols.append("fe_A1C_ordinal")

if "max_glu_serum" in fe_df.columns and fe_df["max_glu_serum"].dtype == "object":
    fe_df["fe_glu_ordinal"] = fe_df["max_glu_serum"].map({"None":0, "Norm":1, ">200":2, ">300":3}).astype("Int64")
    added_cols.append("fe_glu_ordinal")

# 2G) Diagnosis grouping flags from ICD-9 codes in diag_1/2/3
def icd_flag(code, ranges):
    """
    code: raw diag string (e.g., '250.13'); ranges: list of (lo, hi) integer ranges or single ints
    returns 1 if falls into any range, else 0
    """
    try:
        c = str(code)
        if c.startswith('V') or c.startswith('E'):  # skip non-numeric codes
            return 0
        num = int(float(c))  # take integer part (e.g., 250.13 -> 250)
    except:
        return 0
    for lo, hi in ranges:
        if lo <= num <= hi:
            return 1
    return 0

diag_cols = [c for c in ["diag_1","diag_2","diag_3"] if c in fe_df.columns]
if diag_cols:
    ranges = {
        "fe_any_diabetes": [(250, 250)],
        "fe_any_circulatory": [(390,459),(785,785)],
        "fe_any_respiratory": [(460,519)],
        "fe_any_digestive": [(520,579)],
        "fe_any_injury": [(800,999)],
        "fe_any_musculoskeletal": [(710,739)],
        "fe_any_genitourinary": [(580,629)],
        "fe_any_neoplasms": [(140,239)],
    }
    for newcol, rgs in ranges.items():
        fe_df[newcol] = 0
        for d in diag_cols:
            fe_df[newcol] = fe_df[newcol] | fe_df[d].apply(lambda x: icd_flag(x, rgs))
        fe_df[newcol] = fe_df[newcol].astype("Int64")
        added_cols.append(newcol)

shape_note("After feature engineering", fe_df)
print("Added features:", added_cols)
display(fe_df[added_cols].head() if added_cols else fe_df.head())

After feature engineering    -> shape: (101766, 123)
Added features: ['fe_total_visits', 'fe_labs_per_day', 'fe_meds_per_day', 'fe_procs_to_meds', 'fe_any_diabetes', 'fe_any_circulatory', 'fe_any_respiratory', 'fe_any_digestive', 'fe_any_injury', 'fe_any_musculoskeletal', 'fe_any_genitourinary', 'fe_any_neoplasms']


,fe_total_visits,fe_labs_per_day,fe_meds_per_day,fe_procs_to_meds,fe_any_diabetes,fe_any_circulatory,fe_any_respiratory,fe_any_digestive,fe_any_injury,fe_any_musculoskeletal,fe_any_genitourinary,fe_any_neoplasms
0,0,41.000000,1.0,0.000000,0,0,0,0,0,0,0,0
1,0,19.666667,6.0,0.000000,0,0,0,0,0,0,0,0
2,3,5.500000,6.5,0.384615,0,0,0,0,0,0,0,0
3,0,22.000000,8.0,0.062500,0,0,0,0,0,0,0,0
4,0,51.000000,8.0,0.000000,0,0,0,0,0,0,0,0


In [15]:
# =========================
# STEP 3 — SAVE FEATURE-ENGINEERED DATA & SUMMARY
# =========================
out_csv = os.path.join(OUTPUTS_DIR, "diabetic_feature_engineered.csv")
fe_df.to_csv(out_csv, index=False)
print("Saved feature-engineered dataset to:", out_csv)

# Small summary of new columns
summary = pd.DataFrame({
    "new_feature": added_cols,
    "dtype": [str(fe_df[c].dtype) for c in added_cols]
})
summary_path = os.path.join(OUTPUTS_DIR, "feature_summary.csv")
summary.to_csv(summary_path, index=False)
print("Saved feature summary to:", summary_path)

display(summary)

Saved feature-engineered dataset to: C:\Users\dusik\OneDrive\Desktop\Y2S1\Results\Outputs\diabetic_feature_engineered.csv
Saved feature summary to: C:\Users\dusik\OneDrive\Desktop\Y2S1\Results\Outputs\feature_summary.csv


,new_feature,dtype
0,fe_total_visits,int64
1,fe_labs_per_day,float64
2,fe_meds_per_day,float64
3,fe_procs_to_meds,float64
4,fe_any_diabetes,Int64
5,fe_any_circulatory,Int64
6,fe_any_respiratory,Int64
7,fe_any_digestive,Int64
8,fe_any_injury,Int64
9,fe_any_musculoskeletal,Int64
